In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import * 
from pyspark.sql.window import Window

In [0]:
%sql
select * from fintech360.bronze.upi_data;

In [0]:
df = spark.read.table('fintech360.bronze.upi_data')
display(df)

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
    DecimalType, BooleanType, LongType
)

UPI_PARENT_SCHEMA = StructType([
    StructField("upi_txn_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("payer_vpa", StringType(), True),
    StructField("payee_vpa", StringType(), True),
    StructField("merchant_category", StringType(), True),
    StructField("transaction_amount", DecimalType(18, 2), True),
    StructField("currency", StringType(), True),
    StructField("transaction_ts", TimestampType(), True),
    StructField("transaction_status", StringType(), True),
    StructField("rrn", StringType(), True),
    StructField("upi_app", StringType(), True),
    StructField("device_os", StringType(), True),
    StructField("is_refund", BooleanType(), True),
    StructField("created_at", LongType(), True),
    StructField("updated_at", LongType(), True),
    StructField("upi_details_json", StringType(), True),
    StructField("_rescued_data", StringType(), True),
    StructField("_ingested_at", TimestampType(), True),
    StructField("_source_file", StringType(), True)
])

In [0]:
from pyspark.sql import functions as F

def normalize_upi_schema(df, parent_schema=UPI_PARENT_SCHEMA):
    parent_names = [field.name for field in parent_schema.fields]
    parent_fields = {field.name: field for field in parent_schema.fields}

    # Capture unexpected source columns
    extra_cols = [c for c in df.columns if c not in parent_names]

    if extra_cols:
        extra_json = F.to_json(
            F.struct(*[F.col(c) for c in extra_cols])
        )

        if "_rescued_data" in df.columns:
            df = df.withColumn(
                "_rescued_data",
                F.when(
                    F.col("_rescued_data").isNull(),
                    extra_json
                ).otherwise(
                    F.concat_ws("\n", F.col("_rescued_data"), extra_json)
                )
            )
        else:
            df = df.withColumn("_rescued_data", extra_json)

    # Convert numeric-formatted IDs such as 1000.0 to 1000
    if "customer_id" in df.columns:
        value = F.trim(F.col("customer_id").cast("string"))
        numeric_value = value.cast("decimal(38,10)")

        df = df.withColumn(
            "customer_id",
            F.when(
                value.rlike(r"^[+-]?\d+\.0+$"),
                numeric_value.cast("decimal(38,0)").cast("string")
            ).otherwise(value)
        )

    # Add missing fields and cast columns to canonical types
    for name, field in parent_fields.items():
        if name not in df.columns:
            df = df.withColumn(name, F.lit(None).cast(field.dataType))
        else:
            df = df.withColumn(name, F.col(name).cast(field.dataType))

    return df.select(*parent_names)

In [0]:
df = normalize_upi_schema(df)

df.printSchema()
display(df.limit(10))

In [0]:
(df.write
    .mode('append')
    .format('delta')
    .saveAsTable('fintech360.schema_validation.upi_data'))